<p align='center'>
<img src='../../source/_static/logo-light.png' alt='rna-score logo' width='200' align='center'/>
</p>
<p align="center">rna-score</p>


> this page was generated from docs/source/library/user_guide.ipynb <a href="https://colab.research.google.com/github/raysas/structural-RNA-project/blob/dev/lib/docs/source/library/user_guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Demo in colab"/></a>

# User Guide

_this is a step by step user guide to the rna_score library_

**Motivation:** 
The `rna_score` library is designed to provide a toolkit for RNA strcture prediction, particulalry aiming to create a suitable scoring function for RNA 3D structures. It offers a workflow to "train" a statistical potential based on known RNA structures, and then use this potential to score new RNA structures. The library is built to be modular and extensible, allowing users to customize various components of the scoring process.

## Installation

The library can be installed via pip. You can install it directly from the GitHub repository using the following command:

In [2]:
# !pip install git+https://github.com/raysas/structural-RNA-project.git

if installation is broken, please refer to installation in development mode:

<!-- ```bash
git clone https://github.com/your-repo/structural-RNA-project.git
cd structural-RNA-project
pip install -r requirements.txt
pip install -e .
``` -->

In [ ]:
# !git clone https://github.com/your-repo/structural-RNA-project.git
# !cd structural-RNA-project
# !pip install -r requirements.txt
# !pip install -e .

In [1]:
import rna_score
rna_score.__version__

'0.1.1'

As the library is under the `lib` submodule, you can import it as follows for efficient access

In [2]:
import rna_score.lib as rna_score_lib

## Downloading RNA Structures

To start training a scorign function, you first need a dataset of known RNA 3D structures. The library provides utilities to download RNA structures from public databases like the Protein Data Bank (PDB). You can use the `download_rna_structures` function to fetch RNA structures based on specific criteria, and use the `help()` function to get more information about its usage.

We will start off by installing 1000 RNA structures from the PDB database.

In [3]:
rna_score_lib.download_rna_structures(max_structures=100)
print('-- downloaded 100 RNA structures')

Searching for RNA structures in PDB...
Found 100 RNA structures

Output directory: rna_structures


Downloading: 100%|███▉| 199/200 [00:46<00:00,  4.40it/s, Success=199, Failed=0]


Download Summary:
  Total attempted: 200
  Successful: 200
  Failed: 0

PDB IDs saved to: rna_structures/downloaded_ids.txt
-- downloaded 100 RNA structures


Downloading: 100%|████| 200/200 [00:46<00:00,  4.32it/s, Success=200, Failed=0]


## Example 1: Default Scoring Function

### Creating a Scorer object

The score function is encapsulated in a `RNAScorer` class, which manages the entire scoring process. You can create an instance of the `RNAScorer` class which will have the training performed on

In [4]:
vanilla_scorer = rna_score_lib.RNAScorer()

### Extraction of distances

First starting by the extracting distances from structures in either `structures/pdb` or `structures/cif`, as it can read both formats.  
The default parameters for bins, atom_mode and ethod (histogram) are used when no parameters are provided, others can be specified as needed (see examples below).

In [5]:
vanilla_scorer.extract_distances(folder='rna_structures/pdb')

Searching for files in: rna_structures/pdb
Found 100 files to process. Starting batch processing...
Starting Parallel Processing (Atom Mode: C3')...
  ...processed 10/100 structures
  ...processed 20/100 structures
  ...processed 30/100 structures
  ...processed 40/100 structures
  ...processed 50/100 structures
  ...processed 60/100 structures
  ...processed 70/100 structures
  ...processed 80/100 structures
  ...processed 100/100 structures
Batch processing complete in 1.28 seconds.

Writing results to 'dist_data/' (Method: HISTOGRAM)...
Pipeline completed successfully.


PosixPath('dist_data')

### Training the Scoring Function

Here the training following the formula described in the documentation is performed, using the extracted distances to compute the statistical potential (v0.1.0).

In [6]:
vanilla_scorer.train_scoring()

PosixPath('training_output')

We can check the computation's output already performed are stored in the `RNAScorer` object for further use, like the histograms of each atom pair

In [7]:
vanilla_scorer.histograms['AA_histogram']

,count
0,0
1,0
2,0
3,3
4,14
5,33
6,35
7,51
8,76
9,94


### Plotting the Scoring Function

An interactive view of the scorings accross teh different pairs can be visualized via the `plot_scores()` method of the `RNAScorer` class. If you're looking to save the plots, you can use `save_plot_scores()` method, specifying the desired output directory if needed (or will use default folder), as it will generate both png and html for each atom pair.

In [8]:
vanilla_scorer.plot_scores()

The output will be an interactive plotly plot like below: 

![plot_scores_example](../_static/non_smooth_scorings.png)

For KDE:

![plot_scores_kde_example](../_static/scorings.png)

In [9]:
vanilla_scorer.save_plot_scores()

Plotting scoring profiles from training_output …
  ✓ AA
  ✓ AC
  ✓ AG
  ✓ AU
  ✓ CC
  ✓ CG
  ✓ GG
  ✓ CU
  ✓ GU
  ✓ UU

✓ Complete, plots saved in: plots
	plotly interactive plots in "plots/html"
	static png plots in "plots/png"


PosixPath('plots')

### Score a new RNA structure

At the end, this function has been trained to score a new RNA structure with the `score_structure()` method of the `RNAScorer` class. You can provide a path to a new RNA structure file (in PDB or CIF format), and the method will compute the score based on the trained statistical potential.

In [10]:
vanilla_scorer.score_structure(pdb_path='rna_structures/pdb/1ac3.pdb')

Loading scoring tables from training_output...
  Loaded 8 base pair tables

[1/1] Scoring: 1ac3.pdb...

SCORING RESULTS
Structure: 1ac3.pdb
Number of interactions: 5
Total Gibbs free energy estimate: -0.030

Detailed scores saved to: score_results.csv


## Example 2

Trying different parameters:
* mmcif format
* different atom mode
* different binning strategy
* different method (KDE)
* different distance strategy

bold_scorer=RNAScorer()